# Test Data

In [20]:
doc1 = """The Light Running Margin (LRM) is a critical design parameter in naval architecture that accounts for the inevitable increase in resistance a vessel faces over its operational lifetime. 
When a ship is newly built, its hull is smooth and its propeller is clean, allowing it to achieve design speeds with minimum effort. 
However, as the ship ages, factors such as hull fouling (the accumulation of marine growth) and surface roughening due to corrosion significantly increase the power required to maintain the same speed. 
The LRM ensures the engine is calibrated to handle these "heavy" conditions without exceeding its thermal or mechanical limits.
"""
doc2 = """Technically, the LRM is defined as the percentage difference between the Light Running Curve (the theoretical power/speed relationship of a clean hull in calm seas) and the Layout Curve of the engine. 
In practice, a margin is "baked into" the propeller design so that the propeller absorbs less power than the engine's nominal capacity at a given RPM during initial sea trials. 
This intentional mismatch allows the engine to stay within its optimal operating envelope even when the hull becomes fouled or the ship encounters adverse weather and heavy sea states.
"""
doc3 = """Without an adequate LRM, a vessel would eventually face a "power-limited" scenario. As resistance increases over time, the propeller requires more torque to maintain the same revolutions. 
If the margin is too small, the engine may hit its torque limit before reaching its maximum rated speed, leading to a phenomenon known as "heavy running." 
This not only reduces the ship's maximum attainable speed but also increases fuel consumption and places excessive thermal stress on engine components like cylinders and turbochargers.
"""
doc4 = """Current industry standards, influenced by major engine manufacturers like MAN Energy Solutions and WinGD, typically recommend a Light Running Margin between 3% and 7%, depending on the vessel type and its intended trade route. 
For example, ships operating in tropical waters where marine growth is rapid, or those frequently crossing the North Atlantic's rough seas, may require a higher margin. 
This buffer ensures that the engine's "Operating Point" remains safely to the left of the engine's shop-trial curve for as long as possible.
"""
doc5 = """In the modern era of maritime engineering, the LRM also plays a vital role in environmental compliance and EEDI (Energy Efficiency Design Index) ratings. 
By optimizing the margin, designers can ensure the engine operates at its Minimum Specific Fuel Oil Consumption (SFOC) point for a larger portion of its lifespan. 
Effectively, a well-calculated LRM is a balance between initial trial performance and long-term operational efficiency, ensuring the vessel remains reliable, economical, and compliant from its first voyage to its last.
"""

In [24]:
docs = [doc1, doc2, doc3, doc4, doc5]
docs

['The Light Running Margin (LRM) is a critical design parameter in naval architecture that accounts for the inevitable increase in resistance a vessel faces over its operational lifetime. \nWhen a ship is newly built, its hull is smooth and its propeller is clean, allowing it to achieve design speeds with minimum effort. \nHowever, as the ship ages, factors such as hull fouling (the accumulation of marine growth) and surface roughening due to corrosion significantly increase the power required to maintain the same speed. \nThe LRM ensures the engine is calibrated to handle these "heavy" conditions without exceeding its thermal or mechanical limits.\n',
 'Technically, the LRM is defined as the percentage difference between the Light Running Curve (the theoretical power/speed relationship of a clean hull in calm seas) and the Layout Curve of the engine. \nIn practice, a margin is "baked into" the propeller design so that the propeller absorbs less power than the engine\'s nominal capacit

In [11]:
queries = ["what is the range of running rate margine?", "가동률 마진의 범위는 얼마입니까?"]

In [9]:
from FlagEmbedding import BGEM3FlagModel

model = BGEM3FlagModel('../models/bge-m3', use_fp16=True)
model

d:\hybridsearch\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
embeddings_1 = model.encode(docs, batch_size=12, max_length=512,)['dense_vecs']
embeddings_2 = model.encode(queries)['dense_vecs']
similarity = embeddings_1 @ embeddings_2.T
similarity

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


array([[0.53488415, 0.46969727],
       [0.51951957, 0.46726802],
       [0.50177485, 0.44565278],
       [0.5790758 , 0.5696294 ],
       [0.42498794, 0.44768387]], dtype=float32)

In [16]:
output_1 = model.encode(docs, return_dense=True, return_sparse=True, return_colbert_vecs=True)
output_2 = model.encode(queries, return_dense=True, return_sparse=True, return_colbert_vecs=True)

In [17]:
lexical_scores = model.compute_lexical_matching_score(output_1['lexical_weights'], output_2['lexical_weights'])
lexical_scores

array([[0.00067811, 0.        ],
       [0.00028261, 0.        ],
       [0.06775711, 0.        ],
       [0.        , 0.        ],
       [0.00048606, 0.        ]])

In [18]:
sentence_pairs = [[i,j] for i in queries for j in docs]
sentence_pairs

[['what is the range of running rate margine?',
  'The Light Running Margin (LRM) is a critical design parameter in naval architecture that accounts for the inevitable increase in resistance a vessel faces over its operational lifetime. \nWhen a ship is newly built, its hull is smooth and its propeller is clean, allowing it to achieve design speeds with minimum effort. \nHowever, as the ship ages, factors such as hull fouling (the accumulation of marine growth) and surface roughening due to corrosion significantly increase the power required to maintain the same speed. \nThe LRM ensures the engine is calibrated to handle these "heavy" conditions without exceeding its thermal or mechanical limits.\n'],
 ['what is the range of running rate margine?',
  'Technically, the LRM is defined as the percentage difference between the Light Running Curve (the theoretical power/speed relationship of a clean hull in calm seas) and the Layout Curve of the engine. \nIn practice, a margin is "baked int

In [27]:
from pprint import pprint
pprint(model.compute_score(sentence_pairs, 
                          max_passage_length=128, # a smaller max length leads to a lower latency
                          weights_for_different_modes=[0.2, 0.4, 0.4])) # weights_for_different_modes(w) is used to do weighted sum: w[0]*dense_score + w[1]*sparse_score + w[2]*colbert_score


{'colbert': [0.5716255903244019,
             0.5507968068122864,
             0.5644854307174683,
             0.6186769604682922,
             0.4787810444831848,
             0.4776371717453003,
             0.4558548033237457,
             0.4256557822227478,
             0.5829695463180542,
             0.45146432518959045],
 'colbert+sparse+dense': [0.3368251621723175,
                          0.3247789144515991,
                          0.35325199365615845,
                          0.36328595876693726,
                          0.2762579619884491,
                          0.28597456216812134,
                          0.27604421973228455,
                          0.2593928873538971,
                          0.347113698720932,
                          0.27043700218200684],
 'dense': [0.539474368095398,
           0.5217576026916504,
           0.501774787902832,
           0.5790759325027466,
           0.4227185547351837,
           0.4745984971523285,
           0.468511

In [28]:
from pprint import pprint
pprint(model.compute_score(sentence_pairs, 
                          max_passage_length=128, # a smaller max length leads to a lower latency
                          weights_for_different_modes=[0.1, 0.8, 0.1])) # weights_for_different_modes(w) is used to do weighted sum: w[0]*dense_score + w[1]*sparse_score + w[2]*colbert_score


{'colbert': [0.5716255903244019,
             0.5507968068122864,
             0.5644854307174683,
             0.6186769604682922,
             0.4787810444831848,
             0.4776371717453003,
             0.4558548033237457,
             0.4256557822227478,
             0.5829695463180542,
             0.45146432518959045],
 'colbert+sparse+dense': [0.11167006194591522,
                          0.1074727475643158,
                          0.16083171963691711,
                          0.11977528780698776,
                          0.09055361896753311,
                          0.095223568379879,
                          0.09243662655353546,
                          0.08713085949420929,
                          0.11525989323854446,
                          0.09007206559181213],
 'dense': [0.539474368095398,
           0.5217576026916504,
           0.501774787902832,
           0.5790759325027466,
           0.4227185547351837,
           0.4745984971523285,
           0.468